## 概要

こちらで用意した犬と猫の画像400枚（それぞれ200枚ずつ）をデータセットとしてプログラムに取り込み、深層学習を行なってください。
分類器の精度は合否の基準としませんが、正答率がほぼ0％になる場合はプログラムのどこかが間違っているため、不合格の判定になる点に注意してください。

画像のセットを `dog_cat_photos` フォルダに用意しています。これは Flickr のサイトから入手した著作権フリーの犬と猫の写真（すべて画像サイズは75×75）です。

`dog_cat_photos` フォルダには `train` と `test` というフォルダがあり、さらにそのなかにある `dog` および `cat` フォルダ内には、犬と猫それぞれの写真が入っています。`train` の写真は150枚ずつ、`test` の写真は50枚ずつ用意しています。

## 仕様

最終課題用のノートブックのなかに、犬と猫の画像を学習したモデルを作成して、分類を行なうプログラムを作成してください。

- 本レッスン内容で学習した流れに沿って、深層学習プログラムを作成してください。
- データの前処理や水増しの処理を入れてください。
- MobileNetV2 のモデルを利用してください（画像サイズは MobileNetV2 が対応する大きさへのリサイズが必要です）。
- 必ず最後に `evaluate()` を実行して、正答率がわかるようにしてください。

In [5]:
import tensorflow as tf

# 画像の水増しをする関数の定義
def flip_left_right(image, label):   # 左右反転
    image = tf.image.flip_left_right(image)
    return image, label

def flip_up_down(image, label):      # 上下反転
    image = tf.image.flip_up_down(image)
    return image, label

def rot90(image, label):             # 反時計回りに90度回転
    image = tf.image.rot90(image)
    return image, label

def rot180(image, label):            # 反時計回りに180度回転
    image = tf.image.rot90(image, k=2)
    return image, label

def rot270(image, label):            # 反時計回りに270度回転
    image = tf.image.rot90(image, k=3)
    return image, label

# データセットの作成
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/train",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=True
)

# データセットの作成
test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=False
)

# 画像の水増し処理の実行
train_dataset_lr     = train_dataset.map(flip_left_right)
train_dataset_ud     = train_dataset.map(flip_up_down)
train_dataset_rot90  = train_dataset.map(rot90)
train_dataset_rot180 = train_dataset.map(rot180)
train_dataset_rot270 = train_dataset.map(rot270)

# 水増ししたデータセットを結合する
train_dataset = train_dataset.concatenate(train_dataset_lr)
train_dataset = train_dataset.concatenate(train_dataset_ud)
train_dataset = train_dataset.concatenate(train_dataset_rot90)
train_dataset = train_dataset.concatenate(train_dataset_rot180)
train_dataset = train_dataset.concatenate(train_dataset_rot270)

# データをシャッフルする
train_dataset = train_dataset.shuffle(32)

# MobileNetV2モデルを作成する
input_layer = tf.keras.Input(shape=(224, 224, 3))   # 入力層
l_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)   # 前処理（正規化）をする層

base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(224, 224, 3),
    input_tensor=l_layer,
    include_top=False,
    weights="imagenet",
    pooling='avg'
)
base_model.trainable = False   # MobileNetV2の重みを固定する
output_layer = tf.keras.layers.Dense(1, activation='sigmoid')(base_model.output)   # 出力層
model = tf.keras.Model(inputs=input_layer, outputs=output_layer)   # モデルを作成する

# modelに学習させる
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)
model.fit(train_dataset, epochs=20)

# テストデータで分類を実行する
pred_data = model.predict(test_dataset)

# evaluate()でモデルの性能を評価する
model.evaluate(test_dataset)

Found 300 files belonging to 2 classes.
Found 100 files belonging to 2 classes.
Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 73ms/step - accuracy: 0.7761 - loss: 0.4488
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 74ms/step - accuracy: 0.9261 - loss: 0.2240
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.9478 - loss: 0.1734
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.9600 - loss: 0.1395
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 69ms/step - accuracy: 0.9661 - loss: 0.1211
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 71ms/step - accuracy: 0.9706 - loss: 0.1009
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 72ms/step - accuracy: 0.9733 - loss: 0.0920
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.9794 - loss: 0.0791
Epoch 9/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.9833 - loss: 0.0730
Epoch 10/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - accuracy: 0.9872 - loss: 0.0644
Epoch 11/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - accuracy: 0.987

[0.07420685142278671, 0.9599999785423279]